In [ ]:
%load_ext taegis_magic

In [ ]:
%%taegis events search --assign events
FROM process
EARLIEST=-1h | head 5

In [ ]:
from taegis_magic.pandas.events import convert_event_timestamps

In [ ]:
convert_timestamps = events.pipe(convert_event_timestamps)
convert_timestamps[
    [
        "event_time_usec",
        "taegis_magic.event_time_usec",
        "ingest_time_usec",
        "taegis_magic.ingest_time_usec",
    ]
]

In [ ]:
from taegis_magic.pandas.events import inflate_original_data

In [ ]:
inflate_original_data_df = events.pipe(inflate_original_data)
inflate_original_data_df[
    [
        column
        for column in inflate_original_data_df.columns
        if column.startswith("original_data.")
    ]
]

In [ ]:
from taegis_magic.pandas.process import process_correlate_netflow
from pandas import DataFrame

In [ ]:
%%taegis events search --assign netflow_events
FROM netflow 
WHERE processcorrelationid.pid IS NOT NULL AND processcorrelationid.pid contains ':'
EARLIEST=-12h | head 1

In [ ]:
process_events = DataFrame({"process_correlation_id": netflow_events["host_id"] + ":" + netflow_events["processcorrelationid.pid"]})

In [ ]:
tenant = "{insert desired tenant here}"
region = "{insert desired region here}"

In [ ]:
process_netflow_df = process_events.pipe((process_correlate_netflow, "df"), region=region, tenant_id=tenant)